# ClaimScope en Colab

Este notebook ejecuta ClaimScope de punta a punta: descarga un paper de arXiv, extrae sus
afirmaciones, decide cuáles son verificables, diseña un experimento reducido, **lo ejecuta de
verdad** y emite un veredicto con su intervalo de confianza.

## Antes de empezar: el sandbox

En tu máquina, ClaimScope ejecuta el código generado dentro de un contenedor Docker sin red, con
usuario no root y raíz de solo lectura. **Colab no tiene demonio Docker**, así que aquí se usa un
backend más débil: un subproceso con las llamadas de red bloqueadas, la memoria y la CPU limitadas
y el entorno saneado.

Eso **bloquea descargas accidentales pero no contiene código hostil**. Es aceptable aquí porque el
código lo genera un modelo a partir de un paper científico, dentro de una VM desechable de Colab, y
porque el objetivo es ver funcionar la tubería. No lo uses así con código en el que no confías.

El informe que se genera al final lo dice explícitamente.

## 1. Instalación

In [ ]:
# Unos 2 minutos.
!git clone --quiet https://github.com/USUARIO/claimscope.git 2>/dev/null || echo "ya clonado"
%cd claimscope
!pip install --quiet -e . 2>&1 | tail -2
print("listo")

Si no tienes el repositorio en GitHub, sube la carpeta del proyecto a Colab
(icono de carpeta → subir) y salta la celda anterior, ejecutando solo `pip install -e .` sobre
ella.

## 2. La clave del modelo

ClaimScope funciona con Anthropic o con Google Gemini. Gemini tiene un nivel gratuito:
consigue una clave en [aistudio.google.com/apikey](https://aistudio.google.com/apikey).

La clave se pide con `getpass`, así que no queda escrita en el notebook.

In [ ]:
import getpass
import os

provider = "google"  # "google" o "anthropic"

if provider == "google":
    os.environ["GEMINI_API_KEY"] = getpass.getpass("GEMINI_API_KEY: ")
    os.environ["CLAIMSCOPE_MODEL_NAME"] = "gemini-3.6-flash"
else:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")

os.environ["CLAIMSCOPE_PROVIDER"] = provider

# El sandbox de subproceso, porque Colab no tiene Docker.
os.environ["CLAIMSCOPE_SANDBOX_BACKEND"] = "subprocess"

# Un presupuesto pequeño: Colab da CPUs modestas.
os.environ["CLAIMSCOPE_BUDGET_MINUTES_TOTAL"] = "15"
os.environ["CLAIMSCOPE_MAX_CLAIMS"] = "1"

print(f"proveedor: {provider}")

## 3. Comprobar que el sandbox funciona

Antes de gastar llamadas al modelo, verifica que el sandbox ejecuta código y que **bloquea la red
de verdad**. Si esto falla, nada de lo demás tiene sentido.

In [ ]:
from pathlib import Path

from claimscope.config import get_settings
from claimscope.sandbox.runner import ExecutionRequest, build_runner

settings = get_settings()
runner = build_runner(settings)
print(f"backend: {type(runner).__name__}")

ws = Path("/tmp/sandbox_check")
ws.mkdir(exist_ok=True)

# 1. ¿Ejecuta código?
(ws / "run.py").write_text("print('el sandbox ejecuta codigo')")
r = runner.execute(ExecutionRequest(workspace=ws, args=[], timeout_s=30))
print(f"  ejecucion: {'OK' if r.ok else 'FALLO'}  {r.stdout.strip()}")

# 2. ¿Bloquea la red? Esto es lo que importa.
(ws / "run.py").write_text(
    "import socket, sys\n"
    "try:\n"
    "    socket.create_connection(('1.1.1.1', 53), timeout=5)\n"
    "    print('RED ACCESIBLE -- el bloqueo no funciona')\n"
    "except OSError as e:\n"
    "    print(f'red bloqueada: {str(e)[:60]}')\n"
    "    sys.exit(3)\n"
)
r = runner.execute(ExecutionRequest(workspace=ws, args=[], timeout_s=30))
blocked = r.exit_code == 3
print(f"  bloqueo de red: {'OK' if blocked else 'FALLO'}  {r.stdout.strip()}")

assert blocked, "el bloqueo de red no funciona; no continues"
print("\nsandbox verificado")

## 4. Analizar un paper

Elige un paper con una afirmación pequeña y comprobable. El de dropout es ideal: su claim central
es que dropout reduce el sobreajuste en una red pequeña sobre MNIST, y eso cabe en minutos de CPU.

Otras opciones anotadas en el repositorio:

| arXiv | Paper | Por qué |
|---|---|---|
| `1207.0580` | Dropout | Ablación limpia, barata |
| `1708.04552` | Cutout | Una sola perilla, sobre CIFAR |
| `1502.03167` | Batch Normalization | Velocidad de convergencia |
| `1907.11692` | RoBERTa | **Nada verificable** — comprueba que el triage sabe decir que no |

In [ ]:
PAPER = "1207.0580"

import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logging.getLogger("httpx").setLevel(logging.WARNING)

from langgraph.checkpoint.memory import InMemorySaver

from claimscope.graph import build_graph
from claimscope.session import thread_config

graph = build_graph(settings, checkpointer=InMemorySaver())
config = thread_config(f"colab-{PAPER}")

state = graph.invoke({"paper_id": PAPER}, config)

print(f"\n{state.get('paper_title', '')}\n")
for claim in state.get("claims", []):
    mark = "SI " if claim.testable else "no "
    print(f"[{mark}] {claim.claim_type:14} {claim.id}")
    print(f"       {claim.text[:90]}")
    if claim.triage_reason:
        print(f"       -> {claim.triage_reason[:90]}")

print(f"\nseleccionadas: {state.get('selected_claim_ids')}")

## 5. Revisar el plan

El agente se detiene aquí a propósito. Lee el plan de reducción: qué cambia respecto al paper, qué
preserva, y por qué el efecto debería sobrevivir al encogimiento.

**Esta es la parte que más merece tu atención.** Un plan que cambia dos cosas a la vez, o que no
mantiene ambos brazos idénticos salvo en lo que la afirmación dice, produce un resultado que no
significa nada.

In [ ]:
import json

interrupts = state.get("__interrupt__")
if not interrupts:
    print("No hay nada que revisar: el triage no seleccionó ninguna afirmación.")
else:
    for request in interrupts[0].value["requests"]:
        plan = request["plan"]
        print(f"=== {request['claim_id']} ===")
        print(f"{request['claim_text']}\n")
        print(f"Original: {plan['original_setup']}")
        print(f"Reducido: {plan['reduced_setup']}\n")
        print("Cambios:")
        for change in plan["changes"]:
            print(f"  - {change}")
        print("\nPreservado:")
        for item in plan["preserved"]:
            print(f"  - {item}")
        print(f"\nPor que deberia transferirse:\n  {plan['why_claim_should_transfer']}")
        print(f"\nSemillas: {plan['seeds']}  ~{plan['estimated_minutes']:.0f} min")

## 6. Aprobar y ejecutar

Si el plan te convence, apruébalo. A partir de aquí el agente genera el código del experimento, lo
mide con una ejecución corta, comprueba que cabe en el presupuesto y lo ejecuta.

Tarda unos minutos. Si el código generado falla, el nodo `debug` lo arregla y reintenta, hasta
tres veces.

In [ ]:
from langgraph.types import Command

if interrupts:
    decisions = [
        {"claim_id": r["claim_id"], "action": "approve"} for r in interrupts[0].value["requests"]
    ]
    state = graph.invoke(Command(resume=decisions), config)

print("ejecucion terminada\n")
for claim_id, results in (state.get("run_results") or {}).items():
    print(f"{claim_id}: {len(results)} ejecuciones")
    for r in sorted(results, key=lambda x: (x.arm, x.seed)):
        print(f"  {r.arm:20} semilla {r.seed}  metrica {r.metric_value:.4f}  {r.runtime_s:.0f}s")

for claim_id, reason in (state.get("abandoned_claim_ids") or {}).items():
    print(f"\nabandonada {claim_id}: {reason}")

## 7. El código que escribió el modelo

Merece la pena leerlo. Es lo que realmente se ejecutó.

In [ ]:
for claim_id, workspace in (state.get("workspace_dirs") or {}).items():
    script = Path(workspace) / "run.py"
    if script.exists():
        print(f"===== {claim_id} / run.py =====\n")
        print(script.read_text(encoding="utf-8"))

## 8. El veredicto y el informe

El efecto es la diferencia de medias entre brazos, con el signo orientado en la dirección que
predice el paper. El intervalo de confianza sale de un bootstrap sobre las semillas.

- Intervalo entero en la dirección predicha → `consistent_at_reduced_scale`
- Intervalo entero en la contraria → `not_consistent_at_reduced_scale`
- Intervalo que cruza el cero → `inconclusive`

Con tres semillas la potencia es baja: `inconclusive` casi siempre significa "no hay evidencia
suficiente", no "no hay efecto".

In [ ]:
for verdict in state.get("verdicts", []):
    print(f"=== {verdict.claim_id} ===")
    print(f"veredicto: {verdict.verdict}")
    if verdict.effect_estimate is not None:
        print(
            f"efecto:    {verdict.effect_estimate:+.4g}  "
            f"IC95 [{verdict.ci_low:+.4g}, {verdict.ci_high:+.4g}]  p={verdict.p_value:.3g}"
        )
    print(f"{verdict.notes}\n")

report_path = state.get("report_path")
if report_path:
    print(f"\ninforme completo en {report_path}\n")
    print("=" * 70)
    print(Path(report_path).read_text(encoding="utf-8"))

## 9. Qué acabas de comprobar

Si has llegado hasta aquí con un veredicto, has verificado que:

- La ingesta y la extracción de afirmaciones funcionan sobre un paper real.
- El triage distingue lo verificable de lo que no lo es.
- El plan de reducción respeta las invariantes (mismos brazos, mismas semillas, mismo presupuesto).
- El código generado **se ejecuta de verdad** y produce una métrica.
- El análisis estadístico y el informe cierran el ciclo.

Lo que **no** has comprobado: que el sandbox contenga código hostil, porque este backend no puede.
Para eso hace falta Docker.

### Probar el resto

```python
# Los tests de contención del sandbox de subproceso (no necesitan Docker).
!python -m pytest tests/test_subprocess_runner.py -v

# La suite completa.
!python -m pytest -q

# El harness de evaluación sobre los 13 papers anotados.
!python -m eval.run_eval --paper 1207.0580
```